In [ ]:
# Snowsight's Anaconda channel doesn't ship relationalai, so pip install it.
# relationalai 1.2.2 needs protobuf >= 5.27 (for `runtime_version`), but the
# Snowsight runtime pre-imports an older protobuf from /opt/python. Force-
# upgrade protobuf alongside relationalai, then clear any pre-imported
# google.* from sys.modules so the next import picks the new version.
#
# If the next cell still raises ImportError after this runs, click the
# circular arrow icon in the Snowsight toolbar to RESTART the notebook,
# then run again (skipping this cell - pip is already done).
import subprocess, sys, importlib
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
    'protobuf>=5.27,<6', 'relationalai==1.2.2',
])
# Evict any google/protobuf modules that were imported with the old version,
# then invalidate the import cache so the new copy gets picked up.
for _name in list(sys.modules):
    if _name == 'google' or _name.startswith('google.'):
        del sys.modules[_name]
importlib.invalidate_caches()
print('pip install done. If the next cell still hits ImportError, click the')
print('circular arrow (Restart Notebook) in the toolbar above and re-run.')


# EHAM A-CDM Decision Hub - Demo (Snowsight notebook)

Five-act walkthrough of the A-CDM Decision Hub on top of `ACDM_DEMO.EHAM` (320 flights, day of 2026-10-14). Each act uses a different RAI reasoner family on the SAME PyRel ontology.

| Act | Reasoner    | Question |
|-----|-------------|----------|
| 1   | Rules        | TOBT compliance audit (MS12 +/-5 min) |
| 2   | Graph        | KL1234 rotation cascade |
| 3   | Heuristic    | MS5 gate-conflict ranking |
| 4   | Prescriptive | TSAT re-sequence under storm |
| 5   | Persistent rule | Add 'preserve KL high-pax' rule and re-solve |

**This is the Snowsight variant.** PyRel auto-detects the active Snowpark session (`ConfigFromActiveSession`), so no credentials are needed. Sibling .py files (`eham_acdm.py`, `demo_queries.py`) live in the same stage folder as this notebook.

In [ ]:
# Sibling .py modules: eham_acdm.py builds the PyRel model; demo_queries.py
# defines the 5 act queries. The two files travel with this notebook in the
# Snowsight version snapshot AND on the backing stage
# (@ACDM_DEMO.NOTEBOOKS.ACDM_NOTEBOOK_STAGE/planes/), so we probe a handful of
# common locations and fall back to GET'ing them from the stage if needed.
import os, sys, tempfile

_CANDIDATES = [
    os.getcwd(),
    os.path.join(os.getcwd(), "rai_code", "manual"),
    "/home/udf",
    "/snowflake/notebook",
    os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else None,
]
_HERE = next(
    (d for d in _CANDIDATES if d and os.path.isfile(os.path.join(d, "eham_acdm.py"))),
    None,
)

if _HERE is None:
    # Snowsight kernel cwd didn't expose the .py files. Pull them from the
    # stage that backs this notebook. Requires the Snowpark session that
    # Snowsight provides automatically.
    print("sibling .py files not found in cwd; downloading from stage ...")
    from snowflake.snowpark.context import get_active_session
    _session = get_active_session()
    _HERE = tempfile.mkdtemp(prefix="acdm_planes_")
    for _name in ("eham_acdm.py", "demo_queries.py"):
        _session.file.get(
            f"@ACDM_DEMO.NOTEBOOKS.ACDM_NOTEBOOK_STAGE/planes/{_name}",
            _HERE,
        )
    print(f"  downloaded to {_HERE}: {sorted(os.listdir(_HERE))}")

if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

# Disarm PyRel's running-loop guard so synchronous Problem.solve() works
# inside Snowsight (the Snowsight kernel always has an active event loop).
import relationalai.client as _ra_client
import relationalai.services.reasoners.client as _ra_reasoners_client
_noop = lambda *a, **k: None
_ra_client.raise_if_running_event_loop = _noop
_ra_reasoners_client.raise_if_running_event_loop = _noop

# Configure Plotly so figures render inline in Snowsight (Streamlit-backed).

# Snowsight's pyarrow chokes on RAI's Int128Array because the class
# doesn't implement __array__. Add one that downcasts to int64 so the
# internal pyarrow conversion inside .to_df() succeeds.
try:
    from v0.relationalai.clients.result_helpers import Int128Array
    import numpy as _np, pandas as _pd_for_int128
    def _int128_to_int64(self, *_a, **_k):
        return _np.array(
            [0 if v is _pd_for_int128.NA else int(v) for v in self._data],
            dtype='int64',
        )
    Int128Array.__array__ = _int128_to_int64
except Exception as _patch_err:
    print(f'Int128Array patch skipped: {_patch_err}')

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx

# Evict any previously-loaded copy of our sibling modules so a stale
# version doesn't survive Snowsight 'Run all' after a stage upload.
for _m in ('demo_queries', 'eham_acdm'):
    sys.modules.pop(_m, None)

from eham_acdm import (
    Flight, Stand, Operator, GroundHandler, model,
    feeds_callsign, slot_blocks, shares_stand,
    Departure, Arrival, TOBTViolation, StormWindowDeparture, PreservedFlight,
)
from demo_queries import (
    q1_tobt_violations_by_handler,
    q2_rotation_cascade_from,
    q3_ms5_conflict_ranking,
    q4_tsat_resequence_under_storm,
    q5_tsat_resequence_with_preservation,
)
print("imports ok, sibling .py dir:", _HERE)


## Act 1 - Rules: TOBT compliance audit

> **The question (operator types):** 'Show me every flight in the last four hours where actual ready time deviated from TOBT by more than five minutes, broken down by ground handler.'

The `TOBTViolation` derived concept encodes ICAO MS12 once at the ontology layer. The query just counts.

In [ ]:
df1 = q1_tobt_violations_by_handler()
df1

In [ ]:
fig = px.bar(
    df1, x='handler', y='violations',
    color='avg_deviation_min',
    color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
    title='TOBT violations by handler (4h window before 14:30; |ARDT - TOBT| > 5 min)',
    labels={'avg_deviation_min': 'avg deviation (min)'},
    text='violations',
)
fig.update_traces(textposition='outside')
fig.update_layout(height=380, xaxis_title='handler', yaxis_title='violation count')
st.plotly_chart(fig, use_container_width=True)


**What the chart shows.** KLG and AGS are running positive ARDT-vs-TOBT deviations (flights calling ready *late*). DNATA is negative - calling ready *early*, which inflates demand on the TSAT calculator. Both are A-CDM hygiene issues, surfaced from a single derived concept defined once at the ontology layer.

## Act 2 - Graph: KL1234 rotation cascade

> **The question:** 'KL1234 inbound from KJFK is 35 minutes late at final approach. Trace the rotation impact across the next 6 hours.'

The cascade graph is the union of three edge types in the same ontology:
- `feeds_callsign` - explicit aircraft rotation (KL1234 -> KL1235, KL1402 -> KL1601)
- `slot_blocks` - operational pushback contention (KL1235 -> HV5821, KL1402 -> AF1241)
- `shares_stand` - stand-occupancy overlap (E18 carries the KL chain)

The Graph reasoner runs reachability over the union.

In [ ]:
df2 = q2_rotation_cascade_from('KL1234')
df2

In [ ]:
# Cascade graph viz with: carrier-coloured nodes, node size scaled to pax_connections,
# arrow heads per edge, three edge types in distinct colours.
src = Flight.ref(); dst = Flight.ref()
rot = (
    model.where(feeds_callsign(src, dst))
    .select(src.callsign.alias('src'), dst.callsign.alias('dst'))
    .to_df()
)
rot['kind'] = 'rotation'
src = Flight.ref(); dst = Flight.ref()
sb = (
    model.where(slot_blocks(src, dst))
    .select(src.callsign.alias('src'), dst.callsign.alias('dst'))
    .to_df()
)
sb['kind'] = 'slot_block'
src = Flight.ref(); dst = Flight.ref()
ss = (
    model.where(shares_stand(src, dst), src.sobt < dst.sobt)
    .select(src.callsign.alias('src'), dst.callsign.alias('dst'))
    .to_df()
)
ss['kind'] = 'stand'

cascade_callsigns = set(df2['to'])
all_edges = pd.concat([rot, sb, ss], ignore_index=True)
cascade_edges = all_edges[
    all_edges['src'].isin(cascade_callsigns)
    & all_edges['dst'].isin(cascade_callsigns)
].drop_duplicates(subset=['src', 'dst', 'kind'])

# Node metadata
nodes_df = (
    model.where(Flight)
    .select(
        Flight.callsign.alias('callsign'),
        Flight.flight_type.alias('flight_type'),
        Flight.pax_connections.alias('pax_conn'),
    )
    .to_df()
)
nodes_df = nodes_df[nodes_df['callsign'].isin(cascade_callsigns)].copy()
nodes_df['pax_conn'] = nodes_df['pax_conn'].fillna(0).astype('int64')
nodes_df['operator'] = nodes_df['callsign'].str.extract(r'^([A-Z]{2,3})')

G = nx.DiGraph()
for c in cascade_callsigns:
    G.add_node(c)
for _, r in cascade_edges.iterrows():
    G.add_edge(r['src'], r['dst'], kind=r['kind'])
pos = nx.spring_layout(G, seed=42, k=1.7)

edge_styles = {
    'rotation':   {'color': '#1f77b4', 'dash': 'solid', 'width': 3.5},
    'slot_block': {'color': '#d62728', 'dash': 'solid', 'width': 3.5},
    'stand':      {'color': '#8a4fbe', 'dash': 'dot',   'width': 2.0},
}
edge_traces, arrow_anns = [], []
for kind, style in edge_styles.items():
    xs, ys = [], []
    for u, v, data in G.edges(data=True):
        if data.get('kind') != kind:
            continue
        x0, y0 = pos[u]; x1, y1 = pos[v]
        xs += [x0, x1, None]; ys += [y0, y1, None]
        arrow_anns.append(dict(
            ax=x0, ay=y0, x=x1, y=y1,
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=3, arrowsize=1.2, arrowwidth=1.4,
            arrowcolor=style['color'], standoff=18, startstandoff=18, opacity=0.85,
        ))
    if xs:
        edge_traces.append(go.Scatter(
            x=xs, y=ys, mode='lines',
            line=dict(width=style['width'], color=style['color'], dash=style['dash']),
            hoverinfo='none', name=kind, opacity=0.85,
        ))

op_palette = {'KL': '#0064c8', 'HV': '#1aa055', 'AF': '#9c2a2a'}
node_text = list(G.nodes())
node_x = [pos[n][0] for n in node_text]
node_y = [pos[n][1] for n in node_text]
ops = [(nodes_df.loc[nodes_df['callsign'] == n, 'operator'].iloc[0] if (nodes_df['callsign'] == n).any() else '?') for n in node_text]
pax = [int(nodes_df.loc[nodes_df['callsign'] == n, 'pax_conn'].iloc[0]) if (nodes_df['callsign'] == n).any() else 0 for n in node_text]
ftypes = [(nodes_df.loc[nodes_df['callsign'] == n, 'flight_type'].iloc[0] if (nodes_df['callsign'] == n).any() else '?') for n in node_text]
sizes = [max(38, min(82, 38 + p * 0.30)) for p in pax]
node_color = [op_palette.get(o, '#999999') for o in ops]
border_color = ['#ffb000' if n == 'KL1234' else ('#222222' if ft == 'DEPARTURE' else '#1a1a1a') for n, ft in zip(node_text, ftypes)]
border_width = [4 if n == 'KL1234' else (2 if ft == 'DEPARTURE' else 3) for n, ft in zip(node_text, ftypes)]
hover = [f'<b>{n}</b><br>{ft}<br>operator: {o}<br>pax_conn: {p}' for n, ft, o, p in zip(node_text, ftypes, ops, pax)]
node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers+text',
    marker=dict(size=sizes, color=node_color, line=dict(width=border_width, color=border_color)),
    text=node_text, textposition='middle center',
    textfont=dict(size=11, color='#ffffff'),
    name='flight', hoverinfo='text', hovertext=hover,
)
carrier_legend = [
    go.Scatter(x=[None], y=[None], mode='markers',
               marker=dict(size=14, color=c, line=dict(width=1, color='#222')),
               name=op, showlegend=True)
    for op, c in op_palette.items()
]
fig = go.Figure(data=edge_traces + [node_trace] + carrier_legend)
fig.update_layout(
    title='Act 2 - KL1234 cascade (node size = pax connections, colour = carrier)',
    annotations=arrow_anns,
    showlegend=True, legend=dict(orientation='h', x=0.5, y=1.10, xanchor='center'),
    height=560, margin=dict(l=10, r=10, t=60, b=10),
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(visible=False), yaxis=dict(visible=False),
)
st.plotly_chart(fig, use_container_width=True)


**What the chart shows.** From a single late ALDT input (KL1234), the agent reaches six downstream flights across three carriers (KL, HV, AF), two terminals, and one ATFM slot at risk. Edges: blue solid = rotation, red solid = slot_block, purple dotted = stand_share. Gold-ringed node = seed.

## Act 3 - Heuristic: MS5 gate-conflict ranking

> **The question:** 'Looking at all inbound flights with TLDT in the next 30 minutes, rank them by probability of arrival gate conflict at MS5.'

Deterministic fit-score, expressed entirely as derived properties in PyRel:
```
ms5_score = 0.40 * (30 - minutes_to_landing) / 30
          + 0.35 * pax_connections / 150
          + ms5_pier_bonus   (0.15 if pier in {M, G, H} else 0)
          - ms5_wtc_penalty  (0.20 if wtc not in {H, M} else 0)
```

In [ ]:
df3 = q3_ms5_conflict_ranking()
df3

In [ ]:
df3_sorted = df3.copy()
df3_sorted['highlight'] = df3_sorted['inbound'].isin(['DL0036', 'KL0691', 'AF1641', 'BA0432', 'KL1234']).map({True: 'talk-track', False: 'other'})
fig = px.bar(
    df3_sorted.head(11), x='p_conflict', y='inbound', orientation='h',
    color='highlight', color_discrete_map={'talk-track': '#d97a4a', 'other': '#9ec5e8'},
    text='p_conflict',
    hover_data=['current_stand', 'pax_conn', 'wtc', 'minutes_to_land'],
    title='MS5 gate-conflict ranking (talk-track 5 highlighted)',
)
fig.update_traces(texttemplate='%{x:.2f}', textposition='outside')
fig.update_layout(
    height=480,
    yaxis=dict(categoryorder='total ascending', title=None),
    xaxis=dict(title='ms5_score'),
    legend=dict(title='', orientation='h', x=0.5, y=1.05, xanchor='center'),
)
st.plotly_chart(fig, use_container_width=True)


**What the chart shows.** Eleven arrival candidates ranked by their MS5 fit. The five talk-track flights (KL1234, DL0036, KL0691, AF1641, BA0432) all surface. The output is a *ranked queue*, not a yes/no flag.

## Act 4 - Prescriptive: TSAT re-sequence under storm

> **The scenario:** 'Thunderstorm cell crosses 18C/27 from 15:00 to 17:00. 47 outbound flights have TSAT in that window. Solve the optimal sequence.'

Decision variable: binary `FlightSlot.assign_base` over 47 flights x 120 1-minute slots ~5,600 binaries. HiGHS solves in seconds.

Constraints in the Problem: (a) each flight in exactly one slot; (b) each slot at most one flight (single-runway); (c) no push before SOBT; (d) at most 1 simultaneous push per pier; (e) heavy-to-non-heavy needs +1 min wake separation. Objective: minimise total delay weighted by `(1 + 20 * atfm_penalty)`.

In [ ]:
df4, si4 = q4_tsat_resequence_under_storm()
print(f'status: {si4.termination_status}')
print(f'objective: {si4.objective_value:,.0f}')
print(f'solve time: {si4.solve_time_sec:.2f}s')
print(f'flights placed: {len(df4)}')
df4.head(20)

In [ ]:
# Gantt: each flight as a 1-minute bar on the storm window timeline.
df4_gantt = df4.copy()
for col in ('minute_offset', 'abs_min', 'sobt_min', 'delay_min'):
    if col in df4_gantt.columns:
        df4_gantt[col] = df4_gantt[col].astype('int64')
df4_gantt['start'] = pd.Timestamp('2026-10-14 15:00') + pd.to_timedelta(df4_gantt['minute_offset'], unit='m')
df4_gantt['end'] = df4_gantt['start'] + pd.Timedelta(minutes=1)
fig = px.timeline(
    df4_gantt.sort_values('minute_offset'),
    x_start='start', x_end='end', y='callsign',
    color='delay_min', color_continuous_scale='RdYlGn_r',
    hover_data=['op', 'pier', 'pax_conn', 'delay_min'],
    title='Act 4: optimised 18L departure sequence (storm window 15:00 - 17:00)',
)
fig.update_yaxes(categoryorder='total descending')
fig.update_layout(height=900, xaxis_title='time', yaxis_title=None)
st.plotly_chart(fig, use_container_width=True)


In [ ]:
fig = px.histogram(df4, x='delay_min', nbins=30, title='Act 4 delay distribution (min vs SOBT)')
fig.update_layout(height=320)
st.plotly_chart(fig, use_container_width=True)


## Act 5 - Persistent rule: operator adds a preservation rule

> **The operator types:** 'Never delay KL flights with more than 80 connecting pax by more than 8 minutes from SOBT.'

The rule becomes a `PreservedFlight(Flight)` derived concept and an aggregate-delay constraint on the LP. Both live in the same ontology - they persist across solves, not in a runbook.

In [ ]:
df5, si5 = q5_tsat_resequence_with_preservation()
print(f'status: {si5.termination_status}')
print(f'objective: {si5.objective_value:,.0f}')
print(f'solve time: {si5.solve_time_sec:.2f}s')
df5.head(20)

In [ ]:
join = df4[['callsign', 'pax_conn', 'delay_min']].rename(columns={'delay_min': 'act4'}).merge(
    df5[['callsign', 'delay_min']].rename(columns={'delay_min': 'act5'}),
    on='callsign', how='outer',
).fillna(0)
for c in ('act4', 'act5', 'pax_conn'):
    join[c] = join[c].astype('int64')
join['delta'] = (join['act5'] - join['act4']).astype('int64')
moved = join[(join['delta'] != 0) | (join['callsign'] == 'KL691')].sort_values('delta')
moved.head(20)

In [ ]:
long = moved.melt(id_vars=['callsign', 'pax_conn'], value_vars=['act4', 'act5'],
                  var_name='solve', value_name='delay_min')
fig = px.bar(
    long, x='callsign', y='delay_min', color='solve', barmode='group',
    color_discrete_map={'act4': '#9ec5e8', 'act5': '#5cb85c'},
    hover_data=['pax_conn'],
    title='Act 5 - per-flight delay: Act 4 (baseline) vs Act 5 (with preservation rule)',
)
fig.update_layout(
    height=440,
    xaxis=dict(title='callsign', tickangle=-45),
    yaxis=dict(title='delay (min vs SOBT)'),
    legend=dict(orientation='h', x=0.5, y=1.07, xanchor='center', title=''),
)
st.plotly_chart(fig, use_container_width=True)


**Closing.** Five questions, four reasoner families, one semantic model. Adding the Act 5 constraint took ~10 lines of PyRel and didn't break anything else in the model. The rule isn't in a runbook - it lives in the ontology and every reasoner that touches the same model from now on respects it.